# 3章 サンプル（Python版） ― 16×16ドットで「1」を見分ける

256個のマス目（16×16）のON/OFFだけを見て、「数字の1か、そうでないか」を判定する、
いちばん単純な学習器（単層パーセプトロン）。

data2フォルダの10個のファイルをそのまま読み込んでいたオリジナル版（Train2.py）を、
Google Colabでもそのまま動くように、ファイル読み込みをやめてデータをプログラム内に
直接書き込む形（リテラル）に変えている。中身のロジックはオリジナル版と同じ。


In [ ]:
import math, random

# data2フォルダにあった10個のファイルを、そのままリテラルとして持たせたもの
# (ファイル名が"1_"で始まるものが「数字の1」、"0_"で始まるものが「1ではない」)
RAW_DATA = [
    ("0_01.txt", [
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "...■■■■■■■■■■...",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
        "................",
    ]),
    ("0_02.txt", [
        "................",
        "................",
        "....■■■■■■■■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■■■■■■■■....",
        "................",
        "................",
        "................",
        "................",
    ]),
    ("0_03.txt", [
        "................",
        "................",
        "...■............",
        "....■...........",
        ".....■..........",
        "......■.........",
        ".......■........",
        "........■.......",
        ".........■......",
        "..........■.....",
        "...........■....",
        "............■...",
        "................",
        "................",
        "................",
        "................",
    ]),
    ("0_04.txt", [
        "................",
        "................",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        "...■■■■■■■■■....",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        "................",
        "................",
        "................",
    ]),
    ("0_05.txt", [
        "................",
        "................",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "....■......■....",
        "................",
        "................",
        "................",
        "................",
    ]),
    ("1_01.txt", [
        ".......■........",
        "......■■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        "......■■■.......",
        "................",
    ]),
    ("1_02.txt", [
        "......■.........",
        ".....■■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        "......■.........",
        ".....■■■........",
        "................",
    ]),
    ("1_03.txt", [
        "........■.......",
        ".......■■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        "........■.......",
        ".......■■■......",
        "................",
    ]),
    ("1_04.txt", [
        ".......■........",
        ".....■■■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        "......■■■.......",
        "................",
    ]),
    ("1_05.txt", [
        ".......■........",
        "......■■........",
        ".....■.■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".......■........",
        ".....■■■■■......",
        "................",
    ]),
]


## データを読み込む

ファイルの代わりに、上のリテラルデータからSampleを作る。

In [ ]:
class Sample:
    def __init__(self, label, bits, filename):
        self.label, self.bits, self.filename = label, bits, filename


def load_sample(filename, rows):
    if len(rows) != 16 or any(len(row) != 16 for row in rows):
        raise ValueError(filename + " は16文字×16行にしてください")
    bits = [[ch == "■" for ch in row] for row in rows]
    label = filename.startswith("1_")
    return Sample(label, bits, filename)


samples = [load_sample(name, rows) for name, rows in RAW_DATA]


## モデル（重み256個＋bias1個）

In [ ]:
weights = [[0.0]*16 for _ in range(16)]
bias = 0.0

def sigmoid(z):
    return 1.0 / (1.0 + math.exp(-z))

def predict(bits):
    z = bias
    for y in range(16):
        for x in range(16):
            z += (1.0 if bits[y][x] else 0.0) * weights[y][x]
    return sigmoid(z)


In [ ]:
print("===== 学習前 =====")
for s in samples:
    print(s.filename, "正解=", int(s.label), "予測=", round(predict(s.bits), 4))


## 学習ループ

In [ ]:
learning_rate = 0.1
epochs = 1000

for epoch in range(epochs):
    total_loss = 0.0
    random.shuffle(samples)
    for s in samples:
        prediction = predict(s.bits)                 # ①予測
        target = 1.0 if s.label else 0.0
        loss = (prediction - target) ** 2            # ②Loss
        total_loss += loss

        # ③微分: dLoss/dz = 2(p-t) * p(1-p)
        gradient_z = 2*(prediction-target)*prediction*(1-prediction)

        # ④256個のweightを補正
        for y in range(16):
            for x in range(16):
                input_value = 1.0 if s.bits[y][x] else 0.0
                weights[y][x] -= learning_rate * gradient_z * input_value

        # ⑤biasを補正
        bias -= learning_rate * gradient_z

    if epoch == 0 or (epoch+1) % 100 == 0:
        print("epoch =", epoch+1, "loss =", round(total_loss/len(samples), 6))


In [ ]:
print("\n===== 学習後 =====")
for s in samples:
    print(s.filename, "正解=", int(s.label), "予測=", round(predict(s.bits), 4))


## 未知の画像で試す

学習データにはなかった「1」の形を自分で作って、判定させてみる。

In [ ]:
test = [[False]*16 for _ in range(16)]
test[1][7] = True
test[2][6] = test[2][7] = True
for y in range(3,14):
    test[y][7] = True
test[14][6] = test[14][7] = test[14][8] = True

print("===== 未知画像 =====")
for row in test:
    print("".join("■" if v else " " for v in row))

p = predict(test)
print("\n1である予測値 =", round(p,4))
print("判定 =", "1" if p >= 0.5 else "1ではない")
